# modeling_v13 · M4.5 — Lot 효과 진단 (정직축 천장 추정)  ·  v2 (자기완결)

**목적**: M5~M7 기대치·중단 기준 캘리브레이션. "정직축(GKF)에서 얼마나 더 내려갈 수 있나"의 이론적 여지를 챔피언 잔차의 Lot 구조로 추정.

**챔피언(M4 확정)**: Conservative-GA(99피처), B1 = 로컬 GKF 71.366.

> 🔧 **v2 변경(버그픽스)**: v1은 `../문제1(하)/train_data.csv` 를 읽다가 로컬에서 `FileNotFoundError`(한글/OneDrive 경로) 발생.
> → 미리 계산된 `colab_GA/core10_meta_wf.csv` 기반 **자기완결 로더**로 교체. **train_data.csv·modeling_v8 불필요.** 분석 로직·결과 동일(미러 재현 확인).

**전제 파일** (이 노트북은 `modeling_v13/` 안에서 실행):
`colab_GA/v13_select_colab.py`, `colab_GA/v13_fdc_pool_wf_oof.csv.gz`,
`colab_GA/core10_meta_wf.csv`, `colab_GA/feature_diet_selected.json`,
`select_result_Conservative_GA.json`.

**예상 런타임**: 메타 재계산 없음 → GKF-OOF 5 fit(M8_PARAMS·705) = **로컬 ~2~4분·CPU 전용**.

**확인 포인트**:
1. Cell 1 floor assert 통과(R10) · `n_feats=99`.
2. Cell 5 `GKF-OOF RMSE` 가 M4 로컬 **71.366** 을 재현하는지(±0.05) — 재현되면 잔차 분석이 챔피언과 정합.
3. 최종 판독: **잔차 Lot-ICC** 값과 사다리 운영 권고.

> ⚠️ 진단·기록용. 어떤 피처·파라미터도 이 숫자로 고르지 않는다(R1). 정답 파일 미접근(R3).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, sys, json, numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold

# --- 자기완결 로더 (colab_GA/): 풀 + 미리 계산된 core10 메타 병합. train_data.csv·v8 모듈 불필요 ---
sys.path.insert(0, "colab_GA")
import v13_select_colab as sc          # CORE10·M8_PARAMS·BEST_ROUNDS·floor_ok·load(base) 내장

CHAMP = "Conservative"                 # M4 확정 챔피언 = Conservative-GA(99)
df, y, groups, g = sc.load(CHAMP, base="colab_GA")     # colab_GA/ 안의 pool+meta+diet
sel   = json.load(open(f"select_result_{CHAMP}_GA.json", encoding="utf-8"))["selected_features"]
feats = list(dict.fromkeys(g["fixed"] + [f for f in sel if f in df.columns]))   # 15 + 84 = 99
ok, have = sc.floor_ok(feats)
assert ok, f"floor 위반(R10)! {have}"
print(f"챔피언 {CHAMP}-GA : n_feats={len(feats)}  floor={have}")
print(f"df {df.shape} | unique Lot(C20)={df['C20'].nunique()} | WF(C64)={df['C64'].nunique()}")

챔피언 Conservative-GA : n_feats=99  floor={'C17': 4, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 2}
df (11939, 667) | unique Lot(C20)=1217 | WF(C64)=11939


## 진단 1 — C65 Lot 간/내 분산분해 (타깃 구조)

타깃 C65 는 WF 내 상수지만 **Lot 안에서는 WF마다 다르다**. C20(Lot) 기준 일원배치 ICC(1) 로
"C65 변동 중 Lot 소속으로 설명되는 몫" 을 잰다. 높을수록 Lot 단위 구조가 강함.

In [2]:
def icc_oneway(values, grp):
    s = pd.DataFrame({"v": np.asarray(values, float), "g": np.asarray(grp)})
    gs = s.groupby("g")["v"]; n_i = gs.size(); k = len(n_i); N = len(s)
    grand = s["v"].mean()
    ms_b = (n_i * (gs.mean() - grand) ** 2).sum() / (k - 1)
    ss_w = gs.apply(lambda x: ((x - x.mean()) ** 2).sum()).sum()
    ms_w = ss_w / (N - k)
    n0 = (N - (n_i ** 2).sum() / N) / (k - 1)           # 불균등 그룹 보정
    icc = (ms_b - ms_w) / (ms_b + (n0 - 1) * ms_w)
    var_b = max((ms_b - ms_w) / n0, 0.0)
    return dict(icc=float(icc), var_between=float(var_b), var_within=float(ms_w),
                k_lots=int(k), N=int(N), avg_lot=float(N / k))

c65 = icc_oneway(df["C65"], df["C20"])
print(f"C65 Lot-ICC = {c65['icc']:.3f}   (C65 변동 중 Lot 간 몫)")
print(f"  Lot 간 σ ≈ {np.sqrt(c65['var_between']):.1f}  /  Lot 내 σ ≈ {np.sqrt(c65['var_within']):.1f}   (전체 σ_C65=261.7)")
print(f"  Lot 수={c65['k_lots']}  ·  WF={c65['N']}  ·  Lot당 평균 {c65['avg_lot']:.1f} WF")

C65 Lot-ICC = 0.987   (C65 변동 중 Lot 간 몫)
  Lot 간 σ ≈ 260.1  /  Lot 내 σ ≈ 30.0   (전체 σ_C65=261.7)
  Lot 수=1217  ·  WF=11939  ·  Lot당 평균 9.8 WF


## 진단 2 — 챔피언 GKF-OOF 잔차의 Lot-ICC (남은 Lot 효과)

챔피언(M8_PARAMS·705)을 GroupKFold(C20)로 OOF 예측 → 잔차 = 실측 − 예측.
잔차가 **Lot 단위로 뭉치면**(높은 ICC) "센서로 설명 안 된 Lot 효과" 가 남아 있다는 뜻 = 개선 여지.

In [3]:
def gkf_oof(df, feats, y, groups, params, rounds, n_splits=5):
    oof = np.zeros(len(df)); gkf = GroupKFold(n_splits=n_splits)
    for i, (tr, va) in enumerate(gkf.split(df, y, groups=groups), 1):
        m = lgb.train(params, lgb.Dataset(df.iloc[tr][feats], y[tr]), num_boost_round=rounds)
        oof[va] = m.predict(df.iloc[va][feats])
        print(f"  fold {i}/{n_splits} done")
    return oof

oof   = gkf_oof(df, feats, y, groups, sc.M8_PARAMS, sc.BEST_ROUNDS)   # 5 fit, 챔피언 config
resid = y - oof
rmse  = float(np.sqrt(np.mean(resid ** 2)))
print(f"\n챔피언 GKF-OOF RMSE = {rmse:.3f}   (확인포인트 2: M4 로컬 71.366 재현?)")
r_icc = icc_oneway(resid, df["C20"])
print(f"잔차 Lot-ICC = {r_icc['icc']:.3f}")

icc = r_icc["icc"]
if   icc <= 0.2: verdict = "천장 근접 — 남은 Lot 효과 미미. 사다리 축소(F1'만, F4' 스킵). −0.5 달성이 상한일 수 있음"
elif icc >= 0.4: verdict = "여지 큼 — F1'(레짐×센서)·F4'(풀 확장) 적극"
else:            verdict = "중간 — F1' 우선, F4'는 진단3 상관 결과 보고 결정"
print("판독:", verdict)

  fold 1/5 done
  fold 2/5 done
  fold 3/5 done
  fold 4/5 done
  fold 5/5 done

챔피언 GKF-OOF RMSE = 71.366   (확인포인트 2: M4 로컬 71.366 재현?)
잔차 Lot-ICC = 0.784
판독: 여지 큼 — F1'(레짐×센서)·F4'(풀 확장) 적극


## 진단 3 — 잔차–센서 상관 스캔 (F1′·F4′ 기대치 근거)

잔차와 각 피처의 |상관| 상위 → 남은 Lot 효과가 **어떤 센서**와 연동되는지. 이미 챔피언셋에 있으면
교호(F1′)로, 풀 밖 신규 집계면 풀 확장(F4′)으로 흡수 후보.

In [4]:
import re
meta_cols = {"C64", "fold_kf5", "C20", "C65"}
scan = [c for c in df.columns if c not in meta_cols]
rr = pd.Series(resid, index=df.index)
cors = {}
for c in scan:
    v = df[c]
    if v.notna().sum() < 100 or not np.isfinite(v.std(skipna=True)) or v.std(skipna=True) == 0:
        continue
    cors[c] = rr.corr(v)
cs = pd.Series(cors).dropna().sort_values(key=lambda s: s.abs(), ascending=False)
print("잔차와 |상관| 상위 15 피처:")
print(cs.head(15).round(3).to_string())

def sensor(c):
    m = re.match(r"(C\d+)_", c); return m.group(1) if m else c
by = {}
for c, val in cs.items():
    s = sensor(c); by[s] = max(by.get(s, 0.0), abs(val))
bs = pd.Series(by).sort_values(ascending=False)
print("\n센서별 최대 |잔차 상관| 상위 12:")
print(bs.head(12).round(3).to_string())
in_set = set(feats)
print(f"\ntop15 중 이미 챔피언셋 포함: {sum(c in in_set for c in cs.head(15).index)}/15  (많으면 F1' 교호, 적으면 신규정보=F4')")

잔차와 |상관| 상위 15 피처:
C52_std_step6     0.037
C25_std_step6     0.033
C58_max_step5    -0.026
C58_min_step5    -0.026
C58_last_step5   -0.026
C58_mean_step5   -0.026
C58_std_step6     0.025
C52_std_step7     0.022
C57_mean_step5    0.019
C57_min_step5     0.019
C57_last_step5    0.019
C57_max_step5     0.019
C11_min_step5    -0.019
C11_max_step5    -0.019
C11_last_step5   -0.019

센서별 최대 |잔차 상관| 상위 12:
C52    0.037
C25    0.033
C58    0.026
C57    0.019
C11    0.019
C48    0.018
C16    0.018
C60    0.016
C18    0.014
C61    0.014
C56    0.014
C62    0.012

top15 중 이미 챔피언셋 포함: 2/15  (많으면 F1' 교호, 적으면 신규정보=F4')


## 저장 & 천장 추정 요약

In [5]:
res = dict(
    champion=f"{CHAMP}-GA", n_feats=len(feats), gkf_oof_rmse=round(rmse, 3),
    c65_lot_icc=round(c65["icc"], 3),
    c65_sd_between=round(float(np.sqrt(c65["var_between"])), 1),
    c65_sd_within=round(float(np.sqrt(c65["var_within"])), 1),
    resid_lot_icc=round(r_icc["icc"], 3), verdict=verdict,
    top_resid_corr={k: round(float(v), 4) for k, v in cs.head(15).items()},
    sensor_max_corr={k: round(float(v), 4) for k, v in bs.head(12).items()},
)
json.dump(res, open("lot_diagnosis_results.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("saved: lot_diagnosis_results.json")
print(f"\n[천장 추정] 챔피언 잔차 Lot-ICC = {r_icc['icc']:.3f}  →  {verdict}")

saved: lot_diagnosis_results.json

[천장 추정] 챔피언 잔차 Lot-ICC = 0.784  →  여지 큼 — F1'(레짐×센서)·F4'(풀 확장) 적극
